In [584]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns 
from datetime import datetime as dt
import re

df1 = pd.read_excel(r"C:\Users\User\Desktop\Data Analyst\End To End Project\prospecting_sar\Follow up list.xlsx", sheet_name="Sheet1")
df2 = pd.read_excel(r"C:\Users\User\Desktop\Data Analyst\End To End Project\prospecting_sar\Follow up list.xlsx", sheet_name="Sheet2")
df3 = pd.read_excel(r"C:\Users\User\Desktop\Data Analyst\End To End Project\prospecting_sar\Follow up list.xlsx", sheet_name="Sheet3")
df4 = pd.read_excel(r"C:\Users\User\Desktop\Data Analyst\End To End Project\prospecting_sar\Follow up list.xlsx", sheet_name="Sheet4")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
print(df1.shape)
print(df2.shape)
print(df3.shape)
print(df4.shape)

(142, 14)
(52, 14)
(49, 14)
(61, 14)


In [585]:
master = pd.concat([df1, df2,df3,df4], axis=0, ignore_index=True)
master.dtypes

Date               object
First Name         object
Last Name          object
Inquiry            object
DOB                object
Gender             object
Occupation         object
Address            object
Smoking            object
Phone              object
Email              object
Inquiry Time       object
Appt               object
Approach Method    object
dtype: object

# Data Cleaning and EDA

In [586]:

# column renaming
master.columns = master.columns.str.lower()

master.rename(columns={
    "first name": "first_name",
    "last name": "last_name",
    "approach_method": "approach_method",
    "inquiry time": "inquiry_time"
}, inplace=True)




### Date

In [587]:
# date conversion

master["date"] = pd.to_datetime(master["date"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
master["dob"] = pd.to_datetime(master["dob"], format="%d/%m/%Y", errors="coerce")
#master["date"].head()

master["date_approach"] = master["date"].dt.date
master["time_approach"] = master["date"].dt.time
master["time_approach"].head()

0    01:11:44
1    00:05:44
2    04:28:33
3    00:00:49
4    02:50:10
Name: time_approach, dtype: object

In [588]:
master.drop(columns="date", inplace=True)
master.head()

,first_name,last_name,inquiry,dob,gender,occupation,address,smoking,phone,email,inquiry_time,appt,approach method,date_approach,time_approach
0,Muhammad Haziq,Bin Mohd Nor,Hibah,1995-05-29,Lelaki,Tukang masak,Kuala Lumpur,Ya,0167562916,haziqppj@gmail.com,Pagi,Online,WhatsApp,2023-08-02,01:11:44
1,Muhammad Dani,Bin zamani,Kad MedikalHibah,1999-10-20,Lelaki,Kilang,Simpang renggam,Tidak,0172041041,mcoc2064@gmail.com,Tengah Hari,Online,WhatsApp,2023-08-06,00:05:44
2,Zira,Binti rawi,Hibah,1979-11-21,Wanita,Kerani,"NO.6-1-D, BLOK 6, RUMAH PANGSA JALAN HOSPITAL,...",Tidak,0142449493,rawizira4064@gmail.com,Pagi,Online,WhatsApp,2023-08-07,04:28:33
3,Shanizah,Binti Siron,Hibah,1976-11-10,Wanita,Pensyarah,Melaka,Tidak,0166498664,shanizahsiron5506@gmail.com,Petang,Online,WhatsApp,2023-08-08,00:00:49
4,siti marhamah,binti ahmad,Hibah,1988-09-22,Wanita,cleaner,kota tinggi,Tidak,0139094847,sitimarhamah947@gmail.com,Tengah Hari,Online,WhatsApp,2023-08-09,02:50:10


In [589]:
master["date_approach"] = pd.to_datetime(master["date_approach"], format="%Y-%m-%d", errors="coerce")
master["time_approach"] = pd.to_datetime(master["time_approach"], format="%H:%M:%S", errors="coerce").dt.strftime("%H:%M:%S")

current_year = dt.now().year
master["age"] = current_year - master["dob"].dt.year
master.head() 

,first_name,last_name,inquiry,dob,gender,occupation,address,smoking,phone,email,inquiry_time,appt,approach method,date_approach,time_approach,age
0,Muhammad Haziq,Bin Mohd Nor,Hibah,1995-05-29,Lelaki,Tukang masak,Kuala Lumpur,Ya,0167562916,haziqppj@gmail.com,Pagi,Online,WhatsApp,2023-08-02,01:11:44,30.0
1,Muhammad Dani,Bin zamani,Kad MedikalHibah,1999-10-20,Lelaki,Kilang,Simpang renggam,Tidak,0172041041,mcoc2064@gmail.com,Tengah Hari,Online,WhatsApp,2023-08-06,00:05:44,26.0
2,Zira,Binti rawi,Hibah,1979-11-21,Wanita,Kerani,"NO.6-1-D, BLOK 6, RUMAH PANGSA JALAN HOSPITAL,...",Tidak,0142449493,rawizira4064@gmail.com,Pagi,Online,WhatsApp,2023-08-07,04:28:33,46.0
3,Shanizah,Binti Siron,Hibah,1976-11-10,Wanita,Pensyarah,Melaka,Tidak,0166498664,shanizahsiron5506@gmail.com,Petang,Online,WhatsApp,2023-08-08,00:00:49,49.0
4,siti marhamah,binti ahmad,Hibah,1988-09-22,Wanita,cleaner,kota tinggi,Tidak,0139094847,sitimarhamah947@gmail.com,Tengah Hari,Online,WhatsApp,2023-08-09,02:50:10,37.0


### DOB

In [590]:
# Grouping age

bins = [0, 17, 24, 34, 44, 54, 64, 100]
labels = ["Under 18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
master["age_group"] = pd.cut(master["age"], bins=bins, labels=labels, right=True)
print(master.head())

'''
Under 18 — Often excluded or categorized as minors.

18-24 — Young adults (high risk for some types of insurance).

25-34 — Early career stage (still high risk for some insurers).

35-44 — More stable career and family life.

45-54 — Mid-career, typically lower risk.

55-64 — Pre-retirement, may see rising premiums.

65+ — Senior category (often higher premiums for health/life insurance).
'''

       first_name     last_name           inquiry        dob  gender  \
0  Muhammad Haziq  Bin Mohd Nor             Hibah 1995-05-29  Lelaki   
1   Muhammad Dani    Bin zamani  Kad MedikalHibah 1999-10-20  Lelaki   
2            Zira    Binti rawi             Hibah 1979-11-21  Wanita   
3        Shanizah   Binti Siron             Hibah 1976-11-10  Wanita   
4   siti marhamah   binti ahmad             Hibah 1988-09-22  Wanita   

     occupation                                            address smoking  \
0  Tukang masak                                       Kuala Lumpur      Ya   
1        Kilang                                    Simpang renggam   Tidak   
2        Kerani  NO.6-1-D, BLOK 6, RUMAH PANGSA JALAN HOSPITAL,...   Tidak   
3     Pensyarah                                             Melaka   Tidak   
4       cleaner                                        kota tinggi   Tidak   

        phone                        email inquiry_time    appt  \
0  0167562916           haziqpp

'\nUnder 18 — Often excluded or categorized as minors.\n\n18-24 — Young adults (high risk for some types of insurance).\n\n25-34 — Early career stage (still high risk for some insurers).\n\n35-44 — More stable career and family life.\n\n45-54 — Mid-career, typically lower risk.\n\n55-64 — Pre-retirement, may see rising premiums.\n\n65+ — Senior category (often higher premiums for health/life insurance).\n'

### Address

In [591]:

# Sample data (assuming `master["address"]` is your column with address data)
addresses = master["address"].to_list()

# List of states to identify
states = ["Kuala Lumpur", "Selangor", "Melaka", "Sabah", "Penang", "Johor", "Negeri Sembilan", 
          "Terengganu", "Pahang", "Perak", "Kedah", "Kelantan", "Perlis", "Sarawak", "Labuan"]

# Function to extract district and state
def extract_district_state(address):
    # If address is NaN or not a string, return None for district and state
    if not isinstance(address, str):
        return None, None
    
    district = None
    state = None
    
    # Check if any state from the list is in the address
    for s in states:
        if s.lower() in address.lower():
            state = s
            break
    
    # Try to extract district by looking for the first part before the state or end of address
    if state:
        district = address.split(state)[0].strip()
    
    # In case no state was found, leave it as None
    if not state:
        district = address
    return district, state

# Apply the function to each address
districts, states_found = zip(*[extract_district_state(addr) for addr in addresses])

# Create a DataFrame to view the results
df = pd.DataFrame({
    'Address': addresses,
    'District': districts,
    'State': states_found
})

# applying function to master dataframe
master["district"], master["state"] = zip(*[extract_district_state(addr) for addr in master["address"]])    
master.head()


,first_name,last_name,inquiry,dob,gender,occupation,address,smoking,phone,email,inquiry_time,appt,approach method,date_approach,time_approach,age,age_group,district,state
0,Muhammad Haziq,Bin Mohd Nor,Hibah,1995-05-29,Lelaki,Tukang masak,Kuala Lumpur,Ya,0167562916,haziqppj@gmail.com,Pagi,Online,WhatsApp,2023-08-02,01:11:44,30.0,25-34,,Kuala Lumpur
1,Muhammad Dani,Bin zamani,Kad MedikalHibah,1999-10-20,Lelaki,Kilang,Simpang renggam,Tidak,0172041041,mcoc2064@gmail.com,Tengah Hari,Online,WhatsApp,2023-08-06,00:05:44,26.0,25-34,Simpang renggam,None
2,Zira,Binti rawi,Hibah,1979-11-21,Wanita,Kerani,"NO.6-1-D, BLOK 6, RUMAH PANGSA JALAN HOSPITAL,...",Tidak,0142449493,rawizira4064@gmail.com,Pagi,Online,WhatsApp,2023-08-07,04:28:33,46.0,45-54,"NO.6-1-D, BLOK 6, RUMAH PANGSA JALAN HOSPITAL,...",Perak
3,Shanizah,Binti Siron,Hibah,1976-11-10,Wanita,Pensyarah,Melaka,Tidak,0166498664,shanizahsiron5506@gmail.com,Petang,Online,WhatsApp,2023-08-08,00:00:49,49.0,45-54,,Melaka
4,siti marhamah,binti ahmad,Hibah,1988-09-22,Wanita,cleaner,kota tinggi,Tidak,0139094847,sitimarhamah947@gmail.com,Tengah Hari,Online,WhatsApp,2023-08-09,02:50:10,37.0,35-44,kota tinggi,None


In [592]:
master.drop(columns=["district"], inplace=True)
master["state"].value_counts()


state
Johor              31
Selangor           25
Sabah              17
Kedah              13
Pahang             12
Melaka             11
Perak              10
Kuala Lumpur        8
Negeri Sembilan     8
Penang              6
Terengganu          5
Sarawak             5
Kelantan            2
Labuan              1
Perlis              1
Name: count, dtype: int64

### Inquiry

In [593]:

master["inquiry"].value_counts() 
master["inquiry"] = master["inquiry"].str.lower()
master["inquiry"] = master["inquiry"].fillna("other")

# Function to extract key point
def extract_key_point(inquiry):
    if "review" in inquiry:
        return "review"
    elif "hibah" in inquiry:
        return "hibah"
    elif "kad medikal" in inquiry:
        return "kad medikal"
    return "other"  # Return 'other' if no match is found

# Apply the function to the DataFrame
master["inquiry_type"] = master["inquiry"].apply(extract_key_point)
master["inquiry_type"].value_counts()



inquiry_type
hibah          264
review          19
kad medikal     16
other            5
Name: count, dtype: int64

### Policy

In [594]:
master["smoking"].value_counts()

smoking
Tidak              191
Ya                  76
Johor                2
Negeri Sembilan      2
Pulau Pinang         1
Sarawak              1
Perak                1
Pulau Pinang\n       1
Selangor             1
Terengganu           1
Selangor\n           1
Pahang               1
Name: count, dtype: int64

In [595]:


for row in master["smoking"]:
    if row == "Ya":
        master["smoking"].replace(row, "Ya", inplace=True)
    elif row == "Tidak":
        master["smoking"].replace(row, "Tidak", inplace=True)
    else:
        master["smoking"].replace(row, "other", inplace=True)
        
master["smoking"].value_counts()

smoking
Tidak    191
Ya        76
other     37
Name: count, dtype: int64

### Appoinment


In [596]:
master["appt"].value_counts()

appt
Online           232
Depan-depan       35
Pagi              15
Petang             9
Malam              5
Tengah Hari        3
Petang\n           1
Depan-depan\n      1
Name: count, dtype: int64

In [597]:
#master["appt"].value_counts()

# lower case and remove '\n'

master["appt"] = master["appt"].str.lower().str.replace("\n", " ", regex=True)
master["appt"] = master["appt"].str.strip()


for row in master["appt"]:
    if row=="online":
        master["appt"].replace(row, "online", inplace=True)
    else:
        master["appt"].replace(row, "physical", inplace=True)

master["appt"].value_counts()

appt
online      232
physical     72
Name: count, dtype: int64

### Time Approach

In [598]:
print(master["time_approach"].dtype)

master["time_approach"] = pd.to_datetime(master["time_approach"], format="%H:%M:%S", errors="coerce")
master["hour"] = master["time_approach"].dt.hour
master["hour"].value_counts()

# mapping for the hour
def map_time_period(hour):
    if 0 <= hour <= 5:
        return "midnight"
    elif 6 <= hour <= 8:
        return "early_morning"
    elif 9 <= hour <= 11:
        return "morning"
    elif 12 <= hour <= 13:
        return "lunch_time"
    elif 14 <= hour <= 16:
        return "early_afternoon"
    elif 17 <= hour <= 18:
        return "late_afternoon"
    elif 19 <= hour <= 21:
        return "evening"
    elif 22 <= hour <= 23:
        return "night"
    else:
        return "unknown"

master["time_period"] = master["hour"].apply(map_time_period)
master["time_period"].value_counts()


object


time_period
midnight           105
early_morning       52
morning             35
early_afternoon     31
lunch_time          22
late_afternoon      18
unknown             17
night               13
evening             11
Name: count, dtype: int64

### Phone


In [599]:
master["phone"].value_counts()

def clean_phone(phone):
    if phone in ["Ya", "Tidak", "nan"]:
        return "other"
    else:
        return phone

master["phone"] = master["phone"].apply(clean_phone)
master["phone"].value_counts()


phone
other            33
137435632.0       2
129708071.0       2
0167562916        1
135295149         1
194628852         1
1127906560        1
125647977         1
178400759         1
195596916         1
135028184         1
109410133         1
127452999         1
123386504         1
138753400         1
186629679         1
174318020         1
137620437         1
1127826998        1
1111664498        1
1235564554        1
1127354006        1
132502666         1
1125077501        1
105657501         1
134605967         1
127020201         1
168485087         1
197087841         1
1163161055        1
128094711         1
122303615         1
104556163         1
174912916         1
197650085         1
175199700         1
1116139003        1
1137424734.0      1
1139008240.0      1
195291923.0       1
133066411.0       1
133519207.0       1
178652141.0       1
1131986585.0      1
177678341.0       1
195470949.0       1
183991897.0       1
182129897.0       1
1133214950.0      1
183197981.0   

## Finalize data



In [600]:
master.columns
master["time_approach"] = pd.to_datetime(master["time_approach"], format="%H:%M:%S", errors="coerce").dt.strftime("%H:%M:%S")

In [601]:
# rearranging columns
master = master[["first_name", "last_name", "dob", "gender", "phone", "email", "address", "state", "occupation", "smoking", "age", "age_group", "inquiry_type", "appt", "time_approach", "time_period"]]
master.head()

,first_name,last_name,dob,gender,phone,email,address,state,occupation,smoking,age,age_group,inquiry_type,appt,time_approach,time_period
0,Muhammad Haziq,Bin Mohd Nor,1995-05-29,Lelaki,0167562916,haziqppj@gmail.com,Kuala Lumpur,Kuala Lumpur,Tukang masak,Ya,30.0,25-34,hibah,online,01:11:44,midnight
1,Muhammad Dani,Bin zamani,1999-10-20,Lelaki,0172041041,mcoc2064@gmail.com,Simpang renggam,None,Kilang,Tidak,26.0,25-34,hibah,online,00:05:44,midnight
2,Zira,Binti rawi,1979-11-21,Wanita,0142449493,rawizira4064@gmail.com,"NO.6-1-D, BLOK 6, RUMAH PANGSA JALAN HOSPITAL,...",Perak,Kerani,Tidak,46.0,45-54,hibah,online,04:28:33,midnight
3,Shanizah,Binti Siron,1976-11-10,Wanita,0166498664,shanizahsiron5506@gmail.com,Melaka,Melaka,Pensyarah,Tidak,49.0,45-54,hibah,online,00:00:49,midnight
4,siti marhamah,binti ahmad,1988-09-22,Wanita,0139094847,sitimarhamah947@gmail.com,kota tinggi,None,cleaner,Tidak,37.0,35-44,hibah,online,02:50:10,midnight


In [602]:
master.to_excel(r"C:\Users\User\Desktop\Data Analyst\End To End Project\prospecting_sar\master_sar.xlsx", index=False)